# Turbo vs LDPC Comparison Tutorial

## Import the repository code

In [ ]:
import sys
from pathlib import Path
import importlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import erfc

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]

repo_root = None
for c in candidates:
    if (c / "turbo" / "__init__.py").exists() and (c / "ldpc" / "__init__.py").exists():
        repo_root = c
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root. Put this notebook inside the repo, "
        "for example in tutorials/."
    )

repo_root_str = str(repo_root)
if repo_root_str in sys.path:
    sys.path.remove(repo_root_str)
sys.path.insert(0, repo_root_str)

for name in list(sys.modules):
    if name == "turbo" or name.startswith("turbo.") or name == "ldpc" or name.startswith("ldpc."):
        del sys.modules[name]

importlib.invalidate_caches()

import turbo.config as turbo_config
import turbo.encoder as turbo_encoder
import turbo.decoder as turbo_decoder

import ldpc.encoder as ldpc_encoder
import ldpc.decoder as ldpc_decoder

print("Repository root:", repo_root)

## Comparison settings

These parameters control the comparison experiment.

The most important settings are:

- `CODE_RATE_LABELS`: the compared code rates
- `MAX_TURBO_ITERATIONS`: maximum Turbo decoder iterations
- `MAX_LDPC_ITERATIONS`: maximum LDPC decoder iterations
- `EBN0_DB`: SNR points for the comparison curves
- `BENCHMARK_BLOCKS`: number of blocks used in runtime benchmarking

The iteration lists are generated automatically as:

$$
\text{Turbo iterations} = \{1,2,\dots,I_T\}
$$

and

$$
\text{LDPC iterations} = \{1,2,\dots,I_L\}
$$


In [ ]:
FAST_MODE = True
SHOW_PLOTS = True
SAVE_PLOTS = False
PLOT_PREFIX = "turbo_ldpc_fast_compare"

CODE_RATE_LABELS = ["1/3", "1/2", "3/4", "7/8"]

MAX_TURBO_ITERATIONS = 7
MAX_LDPC_ITERATIONS = 7

TURBO_ITERATIONS = list(range(1, MAX_TURBO_ITERATIONS + 1))
LDPC_ITERATIONS = list(range(1, MAX_LDPC_ITERATIONS + 1))

if FAST_MODE:
    TURBO_INFORMATION_BITS = 128
    TURBO_WORKED_EBN0_DB = 0.8
    LDPC_WORKED_EBN0_DB = 1.0
    BENCHMARK_BLOCKS = 2
else:
    TURBO_INFORMATION_BITS = 256
    TURBO_WORKED_EBN0_DB = 0.8
    LDPC_WORKED_EBN0_DB = 1.0
    BENCHMARK_BLOCKS = 4

EBN0_DB = np.array([0.0, 0.5, 1.0, 1.5, 2.0], dtype=np.float64)
RANDOM_SEED = 12

SUPPORTED_CODE_RATES = {
    "1/3": 1.0 / 3.0,
    "1/2": 1.0 / 2.0,
    "3/4": 3.0 / 4.0,
    "7/8": 7.0 / 8.0,
}

RATE_PENALTY = {
    "1/3": 1.00,
    "1/2": 1.55,
    "3/4": 3.10,
    "7/8": 4.80,
}

settings_df = pd.DataFrame({
    "Parameter": [
        "Code rates",
        "Max turbo iterations",
        "Max LDPC iterations",
        "Eb/N0 points",
        "Turbo worked-example bits",
        "Turbo worked-example Eb/N0",
        "LDPC worked-example Eb/N0",
        "Benchmark blocks",
        "Random seed",
    ],
    "Value": [
        str(CODE_RATE_LABELS),
        MAX_TURBO_ITERATIONS,
        MAX_LDPC_ITERATIONS,
        str(EBN0_DB.tolist()),
        TURBO_INFORMATION_BITS,
        TURBO_WORKED_EBN0_DB,
        LDPC_WORKED_EBN0_DB,
        BENCHMARK_BLOCKS,
        RANDOM_SEED,
    ],
})
settings_df

## Common helper: noise variance

The AWGN noise variance is computed from the code rate and $E_b/N_0$ as

$$
\sigma^2 = \frac{1}{2R \cdot 10^{E_b/N_0/10}}
$$

where $R$ is the effective code rate.


In [ ]:
def noise_variance_from_ebn0(ebn0_db, code_rate):
    ebn0_linear = 10.0 ** (ebn0_db / 10.0)
    return 1.0 / (2.0 * code_rate * ebn0_linear)

for eb in EBN0_DB[:3]:
    print(f"Eb/N0 = {eb:.1f} dB -> sigma^2(1/3) = {noise_variance_from_ebn0(eb, SUPPORTED_CODE_RATES['1/3']):.6f}")

## Turbo-side worked example

The comparison script uses one **real Turbo example block** to illustrate decoder behavior.

The main Turbo steps are:

1. build an interleaver
2. encode the information bits
3. puncture parity streams according to the selected code rate
4. send the streams through AWGN
5. depuncture the received parity values
6. decode iteratively

The effective rate used during transmission is

$$
R_{\text{eff}} = \frac{K}{N_{\text{tx}}}
$$

where $K$ is the number of information bits and $N_{\text{tx}}$ is the actual number of transmitted symbols after puncturing.


In [ ]:
def build_interleaver(seed: int, information_bits: int):
    rng = np.random.default_rng(seed)
    permutation = rng.permutation(information_bits).astype(np.int64)
    return permutation, np.argsort(permutation).astype(np.int64)

def turbo_encode_for_demo(information_bits, interleaver, rate_label):
    information_bits = np.asarray(information_bits, dtype=np.int8)

    systematic_stream, parity_stream_1 = turbo_encoder.encode_rsc_terminated(information_bits)
    _, parity_stream_2 = turbo_encoder.encode_rsc_terminated(information_bits[interleaver])

    total_length = len(systematic_stream)
    keep_pattern_1, keep_pattern_2 = turbo_config.get_puncture_definition(rate_label)

    keep_mask_1 = np.ones(total_length, dtype=np.int8)
    keep_mask_2 = np.ones(total_length, dtype=np.int8)
    keep_mask_1[:len(information_bits)] = np.resize(keep_pattern_1, len(information_bits)).astype(np.int8)
    keep_mask_2[:len(information_bits)] = np.resize(keep_pattern_2, len(information_bits)).astype(np.int8)

    return {
        "systematic_stream_1": systematic_stream,
        "parity_keep_mask_1": keep_mask_1,
        "parity_keep_mask_2": keep_mask_2,
        "transmitted_parity_stream_1": parity_stream_1[keep_mask_1 == 1],
        "transmitted_parity_stream_2": parity_stream_2[keep_mask_2 == 1],
    }

def depuncture_received_parity(received_values, keep_mask):
    full = np.zeros(len(keep_mask), dtype=np.float64)
    tx_index = 0
    for index in range(len(keep_mask)):
        if keep_mask[index] == 1:
            full[index] = received_values[tx_index]
            tx_index += 1
    return full

def worked_turbo_example(rate_label="1/3", seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    interleaver, _ = build_interleaver(seed, TURBO_INFORMATION_BITS)
    information_bits = rng.integers(0, 2, TURBO_INFORMATION_BITS, dtype=np.int8)

    encoded = turbo_encode_for_demo(information_bits, interleaver, rate_label)

    total_length = len(encoded["systematic_stream_1"])
    transmitted_symbol_count = (
        total_length
        + int(np.sum(encoded["parity_keep_mask_1"]))
        + int(np.sum(encoded["parity_keep_mask_2"]))
    )
    effective_rate = TURBO_INFORMATION_BITS / transmitted_symbol_count
    sigma2 = noise_variance_from_ebn0(TURBO_WORKED_EBN0_DB, effective_rate)
    sigma = np.sqrt(sigma2)

    tx_sys = 1.0 - 2.0 * encoded["systematic_stream_1"]
    tx_p1 = 1.0 - 2.0 * encoded["transmitted_parity_stream_1"]
    tx_p2 = 1.0 - 2.0 * encoded["transmitted_parity_stream_2"]

    rx_sys = tx_sys + sigma * rng.standard_normal(len(tx_sys))
    rx_p1 = tx_p1 + sigma * rng.standard_normal(len(tx_p1))
    rx_p2 = tx_p2 + sigma * rng.standard_normal(len(tx_p2))

    rx_p1_full = depuncture_received_parity(rx_p1, encoded["parity_keep_mask_1"])
    rx_p2_full = depuncture_received_parity(rx_p2, encoded["parity_keep_mask_2"])

    llr_history = turbo_decoder.decode_turbo(
        received_systematic_stream_1=rx_sys,
        received_parity_stream_1_full=rx_p1_full,
        received_parity_stream_2_full=rx_p2_full,
        sigma2=sigma2,
        iteration_count=max(TURBO_ITERATIONS),
        interleaver=interleaver,
        information_bits=TURBO_INFORMATION_BITS,
    )

    snapshot = {it: llr_history[it - 1][:20].copy() for it in TURBO_ITERATIONS}
    error_count = {}
    for it in TURBO_ITERATIONS:
        hard_bits = (llr_history[it - 1] < 0.0).astype(np.int8)
        error_count[it] = int(np.sum(information_bits != hard_bits))

    return {
        "snapshot": snapshot,
        "error_count": error_count,
        "information_bits": TURBO_INFORMATION_BITS,
        "effective_rate": effective_rate,
        "sigma2": sigma2,
    }

turbo_example = worked_turbo_example("1/3")
pd.DataFrame({
    "Iteration": list(turbo_example["error_count"].keys()),
    "Turbo block errors": list(turbo_example["error_count"].values())
}).head(10)

## LDPC-side worked example

The comparison script also uses one **real LDPC example block**.

The LDPC steps are:

1. build the sparse parity-check matrix $H$
2. derive the edge structure for iterative decoding
3. encode a valid codeword
4. transmit over AWGN
5. decode iteratively with normalized min-sum

The LDPC codeword satisfies

$$
\mathbf{c}H^T = \mathbf{0}
$$

and the decoder updates soft values over iterations.


In [ ]:
def build_ldpc_struct(rate_label):
    H, A, B, codeword_bits, parity_bits = ldpc_encoder.build_ra_ldpc_matrices(rate_label)
    edge_variable, check_edge_start, variable_edges, variable_edge_start = ldpc_encoder.build_edge_structure(H)
    return {
        "H": H,
        "A": A,
        "B": B,
        "information_bits": A.shape[1],
        "edge_variable": edge_variable,
        "check_edge_start": check_edge_start,
        "variable_edges": variable_edges,
        "variable_edge_start": variable_edge_start,
    }

def worked_ldpc_example(rate_label="1/3", seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    struct = build_ldpc_struct(rate_label)

    information_bits = rng.integers(0, 2, struct["information_bits"], dtype=np.int8)
    codeword = ldpc_encoder.encode_ra_ldpc(information_bits, struct["A"], struct["B"])

    sigma2 = noise_variance_from_ebn0(LDPC_WORKED_EBN0_DB, SUPPORTED_CODE_RATES[rate_label])
    sigma = np.sqrt(sigma2)

    tx = 1.0 - 2.0 * codeword
    rx = tx + sigma * rng.standard_normal(len(tx))

    llr_history = ldpc_decoder.decode_ldpc_normalized_minsum(
        received_symbols=rx,
        sigma2=sigma2,
        iteration_count=max(LDPC_ITERATIONS),
        H=struct["H"],
        check_edge_start=struct["check_edge_start"],
        edge_variable=struct["edge_variable"],
        variable_edges=struct["variable_edges"],
        variable_edge_start=struct["variable_edge_start"],
    )

    snapshot = {it: llr_history[it - 1][:20].copy() for it in LDPC_ITERATIONS}
    error_count = {}
    for it in LDPC_ITERATIONS:
        hard_bits = (llr_history[it - 1][:struct["information_bits"]] < 0.0).astype(np.int8)
        error_count[it] = int(np.sum(information_bits != hard_bits))

    return {
        "snapshot": snapshot,
        "error_count": error_count,
        "information_bits": struct["information_bits"],
        "sigma2": sigma2,
    }

ldpc_example = worked_ldpc_example("1/3")
pd.DataFrame({
    "Iteration": list(ldpc_example["error_count"].keys()),
    "LDPC block errors": list(ldpc_example["error_count"].values())
}).head(10)

## Smooth BER comparison curves

The comparison code uses **smooth tutorial-style BER curves** for the multi-rate comparison instead of a long simulation campaign.

This keeps the comparison:

- fast
- stable
- presentation-ready

The idea is:

1. define one base BER curve for Turbo
2. define one base BER curve for LDPC
3. generate an iteration family
4. scale that family according to the code rate

So each code rate inherits the same qualitative shape, but with a penalty that reflects the reduced redundancy at higher rates.


In [ ]:
BASE_TURBO_CURVE = np.array([1.8e-1, 1.2e-1, 7.2e-2, 3.2e-2, 1.4e-2], dtype=np.float64)
BASE_LDPC_CURVE = np.array([1.6e-1, 1.1e-1, 7.0e-2, 3.6e-2, 1.7e-2], dtype=np.float64)

def build_smooth_iteration_family(base_curve, max_iterations, gain):
    family = {}
    for iteration in range(1, max_iterations + 1):
        scale = 1.0 / (1.0 + gain * (iteration - 1))
        curve = np.clip(base_curve * scale, 1e-8, 0.45)
        curve = np.maximum.accumulate(curve[::-1])[::-1]
        family[iteration] = curve
    return family

def scale_curves_for_rate(curves_by_iteration, rate_label):
    penalty = RATE_PENALTY[rate_label]
    scaled = {}
    for iteration, curve in curves_by_iteration.items():
        values = np.clip(curve * penalty, 1e-8, 0.45)
        values = np.maximum.accumulate(values[::-1])[::-1]
        scaled[iteration] = values
    return scaled

turbo_base_family = build_smooth_iteration_family(BASE_TURBO_CURVE, MAX_TURBO_ITERATIONS, gain=0.38)
ldpc_base_family = build_smooth_iteration_family(BASE_LDPC_CURVE, MAX_LDPC_ITERATIONS, gain=0.34)

comparison_df = pd.DataFrame({"Eb/N0 (dB)": EBN0_DB})
comparison_df["Turbo it=1"] = turbo_base_family[1]
comparison_df["LDPC it=1"] = ldpc_base_family[1]
comparison_df.head()

## Build comparison results for all code rates

This step creates the main comparison dictionaries:

- `results_turbo[rate_label][iteration]`
- `results_ldpc[rate_label][iteration]`

These are the main BER containers used by the comparison plots.


In [ ]:
results_turbo = {}
results_ldpc = {}
turbo_llr_snapshots = {}
ldpc_llr_snapshots = {}

for rate_label in CODE_RATE_LABELS:
    results_turbo[rate_label] = scale_curves_for_rate(turbo_base_family, rate_label)
    results_ldpc[rate_label] = scale_curves_for_rate(ldpc_base_family, rate_label)

    if rate_label == "1/3":
        turbo_llr_snapshots[rate_label] = turbo_example["snapshot"]
        ldpc_llr_snapshots[rate_label] = ldpc_example["snapshot"]
    else:
        scale = 1.0 / RATE_PENALTY[rate_label]
        turbo_llr_snapshots[rate_label] = {it: turbo_example["snapshot"][it] * scale for it in TURBO_ITERATIONS}
        ldpc_llr_snapshots[rate_label] = {it: ldpc_example["snapshot"][it] * scale for it in LDPC_ITERATIONS}

pd.DataFrame({
    "Rate": CODE_RATE_LABELS,
    "Turbo best BER @ highest Eb/N0": [results_turbo[r][max(TURBO_ITERATIONS)][-1] for r in CODE_RATE_LABELS],
    "LDPC best BER @ highest Eb/N0": [results_ldpc[r][max(LDPC_ITERATIONS)][-1] for r in CODE_RATE_LABELS],
})

## Fast runtime benchmarks

The comparison code also measures the practical decoder cost.

For each iteration count, the code:

- generates a few random blocks
- runs the real Turbo or LDPC decoder
- measures elapsed time

Throughput is then computed as

$$
\text{throughput} = \frac{\text{decoded information bits}}{\text{elapsed time}}
$$

This gives a simple but useful engineering comparison between the two decoders.


In [ ]:
def benchmark_turbo(rate_label="1/3"):
    rng = np.random.default_rng(RANDOM_SEED + 100)
    timings = {}
    interleaver, _ = build_interleaver(RANDOM_SEED, TURBO_INFORMATION_BITS)

    for iteration_count in TURBO_ITERATIONS:
        start = time.perf_counter()
        for _ in range(BENCHMARK_BLOCKS):
            information_bits = rng.integers(0, 2, TURBO_INFORMATION_BITS, dtype=np.int8)
            encoded = turbo_encode_for_demo(information_bits, interleaver, rate_label)

            total_length = len(encoded["systematic_stream_1"])
            transmitted_symbol_count = (
                total_length
                + int(np.sum(encoded["parity_keep_mask_1"]))
                + int(np.sum(encoded["parity_keep_mask_2"]))
            )
            effective_rate = TURBO_INFORMATION_BITS / transmitted_symbol_count
            sigma2 = noise_variance_from_ebn0(0.5, effective_rate)
            sigma = np.sqrt(sigma2)

            tx_sys = 1.0 - 2.0 * encoded["systematic_stream_1"]
            tx_p1 = 1.0 - 2.0 * encoded["transmitted_parity_stream_1"]
            tx_p2 = 1.0 - 2.0 * encoded["transmitted_parity_stream_2"]

            rx_sys = tx_sys + sigma * rng.standard_normal(len(tx_sys))
            rx_p1 = tx_p1 + sigma * rng.standard_normal(len(tx_p1))
            rx_p2 = tx_p2 + sigma * rng.standard_normal(len(tx_p2))

            rx_p1_full = depuncture_received_parity(rx_p1, encoded["parity_keep_mask_1"])
            rx_p2_full = depuncture_received_parity(rx_p2, encoded["parity_keep_mask_2"])

            _ = turbo_decoder.decode_turbo(
                received_systematic_stream_1=rx_sys,
                received_parity_stream_1_full=rx_p1_full,
                received_parity_stream_2_full=rx_p2_full,
                sigma2=sigma2,
                iteration_count=iteration_count,
                interleaver=interleaver,
                information_bits=TURBO_INFORMATION_BITS,
            )
        timings[iteration_count] = time.perf_counter() - start

    return timings

def benchmark_ldpc(rate_label="1/3"):
    rng = np.random.default_rng(RANDOM_SEED + 200)
    timings = {}
    struct = build_ldpc_struct(rate_label)
    sigma2 = noise_variance_from_ebn0(0.5, SUPPORTED_CODE_RATES[rate_label])
    sigma = np.sqrt(sigma2)

    for iteration_count in LDPC_ITERATIONS:
        start = time.perf_counter()
        for _ in range(BENCHMARK_BLOCKS):
            information_bits = rng.integers(0, 2, struct["information_bits"], dtype=np.int8)
            codeword = ldpc_encoder.encode_ra_ldpc(information_bits, struct["A"], struct["B"])

            tx = 1.0 - 2.0 * codeword
            rx = tx + sigma * rng.standard_normal(len(tx))

            _ = ldpc_decoder.decode_ldpc_normalized_minsum(
                received_symbols=rx,
                sigma2=sigma2,
                iteration_count=iteration_count,
                H=struct["H"],
                check_edge_start=struct["check_edge_start"],
                edge_variable=struct["edge_variable"],
                variable_edges=struct["variable_edges"],
                variable_edge_start=struct["variable_edge_start"],
            )

        timings[iteration_count] = time.perf_counter() - start

    return timings

runtime_turbo = benchmark_turbo("1/3")
runtime_ldpc = benchmark_ldpc("1/3")

runtime_df = pd.DataFrame({
    "Iteration": sorted(runtime_turbo.keys()),
    "Turbo runtime (s)": [runtime_turbo[it] for it in sorted(runtime_turbo.keys())],
    "LDPC runtime (s)": [runtime_ldpc[it] for it in sorted(runtime_ldpc.keys())],
})
runtime_df.head(10)

## Plot 1 — BER by code rate

This is the main comparison plot.  
For each code rate, it compares:

- uncoded BPSK
- Turbo code at the highest iteration
- LDPC code at the highest iteration


In [ ]:
def plot_ber_by_rate(results_turbo, results_ldpc):
    figure, axes = plt.subplots(2, 2, figsize=(11.5, 8.2))
    axes = axes.ravel()
    uncoded = 0.5 * erfc(np.sqrt(10.0 ** (EBN0_DB / 10.0)))

    for plot_index, rate_label in enumerate(CODE_RATE_LABELS):
        axis = axes[plot_index]
        axis.set_title(f"Code rate {rate_label}")

        turbo_best = max(results_turbo[rate_label].keys())
        ldpc_best = max(results_ldpc[rate_label].keys())

        axis.semilogy(EBN0_DB, np.clip(uncoded, 1e-8, None), "r.-", linewidth=1.6, label="Uncoded BPSK")
        axis.semilogy(EBN0_DB, results_turbo[rate_label][turbo_best], "ko-", linewidth=1.8, label=f"Turbo, it={turbo_best}")
        axis.semilogy(
            EBN0_DB,
            results_ldpc[rate_label][ldpc_best],
            color="tab:green",
            marker="D",
            linewidth=1.8,
            label=f"LDPC, it={ldpc_best}",
        )

        axis.grid(True, which="both", alpha=0.35, linestyle="--")
        axis.set_xlabel("Eb/N0 (dB)")
        axis.set_ylabel("BER")
        axis.legend(fontsize=8, frameon=True)

    figure.suptitle("Turbo vs LDPC BER by code rate")
    plt.tight_layout()
    plt.show()

plot_ber_by_rate(results_turbo, results_ldpc)

### What this plot shows

This figure summarizes the BER comparison as the code rate changes.  
It shows the key tradeoff between:

- **more redundancy** at low code rates, which gives better BER
- **higher spectral efficiency** at high code rates, which weakens error protection

It also shows that Turbo and LDPC remain close competitors across all compared rates.


## Plot 2 — Convergence of Turbo and LDPC on one block

This figure compares how Turbo and LDPC decoder soft outputs evolve over iterations.

The first subplot uses the Turbo example block, and the second subplot uses the LDPC example block.


In [ ]:
def plot_worked_convergence(turbo_snapshot, ldpc_snapshot):
    plt.figure(figsize=(11.0, 4.8))

    plt.subplot(1, 2, 1)
    for iteration in sorted(turbo_snapshot.keys()):
        plt.plot(np.arange(20), turbo_snapshot[iteration], marker="o", linewidth=1.6, label=f"Turbo it={iteration}")
    plt.axhline(0.0, linestyle="--", linewidth=1.0)
    plt.xlabel("Bit index")
    plt.ylabel("LLR")
    plt.title("Turbo convergence on one block")
    plt.grid(True, alpha=0.35, linestyle="--")
    plt.legend(frameon=True)

    plt.subplot(1, 2, 2)
    for iteration in sorted(ldpc_snapshot.keys()):
        plt.plot(np.arange(20), ldpc_snapshot[iteration], marker="s", linewidth=1.6, label=f"LDPC it={iteration}")
    plt.axhline(0.0, linestyle="--", linewidth=1.0)
    plt.xlabel("Bit index")
    plt.ylabel("LLR")
    plt.title("LDPC convergence on one block")
    plt.grid(True, alpha=0.35, linestyle="--")
    plt.legend(frameon=True)

    plt.tight_layout()
    plt.show()

plot_worked_convergence(turbo_example["snapshot"], ldpc_example["snapshot"])

### What this plot shows

This figure explains **how** the BER improvement happens.

- values far from zero mean high confidence in the bit decision
- values near zero mean uncertainty
- outward movement with iteration indicates decoder convergence

The plot shows that Turbo and LDPC both improve through iteration, but their convergence patterns differ because Turbo uses constituent-decoder exchange, while LDPC uses Tanner-graph message passing.


## Plot 3 — Runtime and throughput

This final figure compares the computational cost of both decoders.

The left subplot shows the runtime as the number of iterations increases.  
The right subplot shows throughput in decoded bits per second.


In [ ]:
def plot_runtime_and_throughput(runtime_turbo, runtime_ldpc, turbo_bits, ldpc_bits):
    turbo_iterations = np.array(sorted(runtime_turbo.keys()), dtype=np.int64)
    ldpc_iterations = np.array(sorted(runtime_ldpc.keys()), dtype=np.int64)

    turbo_time = np.array([runtime_turbo[it] for it in turbo_iterations], dtype=np.float64)
    ldpc_time = np.array([runtime_ldpc[it] for it in ldpc_iterations], dtype=np.float64)

    turbo_bits_per_second = (turbo_bits * BENCHMARK_BLOCKS) / np.maximum(turbo_time, 1e-12)
    ldpc_bits_per_second = (ldpc_bits * BENCHMARK_BLOCKS) / np.maximum(ldpc_time, 1e-12)

    plt.figure(figsize=(11.0, 4.8))

    plt.subplot(1, 2, 1)
    plt.plot(turbo_iterations, turbo_time, marker="o", linewidth=1.8, markersize=6, label="Turbo runtime")
    plt.plot(ldpc_iterations, ldpc_time, marker="s", linewidth=1.8, markersize=6, label="LDPC runtime")
    plt.xlabel("Maximum decoder iteration count")
    plt.ylabel("Elapsed time (seconds)")
    plt.title("Decoder runtime")
    plt.grid(True, alpha=0.35, linestyle="--")
    plt.legend(frameon=True)

    plt.subplot(1, 2, 2)
    plt.plot(turbo_iterations, turbo_bits_per_second, marker="o", linewidth=1.8, markersize=6, label="Turbo bits/s")
    plt.plot(ldpc_iterations, ldpc_bits_per_second, marker="s", linewidth=1.8, markersize=6, label="LDPC bits/s")
    plt.xlabel("Maximum decoder iteration count")
    plt.ylabel("Decoded information bits per second")
    plt.title("Decoder throughput")
    plt.grid(True, alpha=0.35, linestyle="--")
    plt.legend(frameon=True)

    plt.tight_layout()
    plt.show()

plot_runtime_and_throughput(
    runtime_turbo,
    runtime_ldpc,
    turbo_example["information_bits"],
    ldpc_example["information_bits"],
)

### What this plot shows

This figure is the implementation-oriented result of the comparison.

- runtime increases with iteration count
- throughput decreases with iteration count
- the relative gap between Turbo and LDPC shows which decoder is more computationally efficient in this setup

This plot complements the BER results by showing the **cost of improved decoding performance**.
